# Lab Day 10 — Qwen3-VL trên Kaggle (bản chạy lại an toàn)

Notebook chia mỗi thí nghiệm thành một cell riêng. Sau E0/E1/E2/E3, hãy tải file backup và bấm **Save Version** trước khi chạy bước tiếp theo. Không dùng **Run All** cho toàn bộ notebook.


## 0. Khởi tạo repository, môi trường và kiểm thử

Bật **GPU** và **Internet** trong Settings. Cell này tự clone/pull fork cá nhân, không cần Kaggle secret. Test template REPORT được loại riêng vì nó yêu cầu còn `<điền>`, trái với checker cuối bài.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/ngocanhpham-hust/K4-Track-4-Day10-VisionLanguageModel.git"
REPO_DIR = Path("/kaggle/working/K4-Track-4-Day10-VisionLanguageModel")

subprocess.run(["nvidia-smi", "-L"], check=True)
if (REPO_DIR / ".git").exists():
    print("Dùng repository hiện có:", REPO_DIR)
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Working directory:", Path.cwd())

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt", "kagglehub"],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "-k", "not test_report_template_has_student_markers_and_ten_questions",
    ],
    check=True,
)


## 1. Tải dữ liệu public và tạo split tất định

Notebook ưu tiên dataset đã gắn trong `/kaggle/input`; nếu chưa có, `kagglehub` tự tải.


In [ ]:
from pathlib import Path
import subprocess
import sys

DATASET_SLUG = "lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes"

def find_dataset_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for path in [base] + [p for p in base.rglob("*") if p.is_dir()]:
        if (path / "images").is_dir() and (path / "labels-YOLO").is_dir():
            return path
    return None

if Path("data/splits/meta.json").exists():
    print("Split đã tồn tại, bỏ qua download.")
else:
    source_root = find_dataset_root("/kaggle/input")
    if source_root is None:
        import kagglehub
        source_root = find_dataset_root(kagglehub.dataset_download(DATASET_SLUG))
    if source_root is None:
        raise FileNotFoundError("Không tìm thấy images/ và labels-YOLO/.")
    print("Dataset source:", source_root)
    subprocess.run(
        [sys.executable, "scripts/prepare_data.py", "--source", str(source_root)],
        check=True,
    )


## 2. Nạp mã lab và công cụ backup

Mỗi backup chứa artifact, prediction và evidence của đúng stage. Hãy tải ZIP xuống máy ngay sau khi stage hoàn thành.


In [ ]:
import importlib
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

import lab_day10_solution
importlib.reload(lab_day10_solution)
import finish_kaggle
importlib.reload(finish_kaggle)

from lab_day10_solution import (
    PROMPT_NORM1000, check_reward_cases, collate_fn, dataset_summary,
    encode_example, image_path, objects_to_answer, read_jsonl,
    sample_visualizations, vision_token_count,
)

def make_stage_backup(stage):
    stage = stage.lower()
    candidates = [
        REPO_DIR / "work" / stage,
        REPO_DIR / "work" / f"{stage}.jsonl",
        REPO_DIR / "report/evidence" / f"{stage}.json",
    ]
    existing = [p for p in candidates if p.exists()]
    if not existing:
        raise FileNotFoundError(f"Chưa có artifact cho {stage}")
    archive = Path(f"/kaggle/working/backup-{stage}.zip")
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in existing:
            if path.is_dir():
                for file in path.rglob("*"):
                    if file.is_file():
                        zf.write(file, file.relative_to(REPO_DIR))
            else:
                zf.write(path, path.relative_to(REPO_DIR))
    print("Đã tạo backup:", archive)
    display(FileLink(str(archive)))
    return archive


def restore_stage_backup(zip_path):
    zip_path = Path(zip_path)
    if not zip_path.exists():
        raise FileNotFoundError(zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(REPO_DIR)
    print("Đã khôi phục:", zip_path)


## 3. Kiểm tra split và xem 6 ảnh ground truth


In [ ]:
import json
summary = dataset_summary()
display(summary)
expected = {
    "train": {"images": 1422, "boxes": {"pothole": 1074, "crack": 1750, "manhole": 691}},
    "val": {"images": 276, "boxes": {"pothole": 66, "crack": 470, "manhole": 103}},
    "test": {"images": 311, "boxes": {"pothole": 121, "crack": 299, "manhole": 163}},
}
for split, values in expected.items():
    assert summary[split]["images"] == values["images"]
    assert summary[split]["boxes"] == values["boxes"]
meta = json.loads(Path("data/splits/meta.json").read_text())
assert meta["dataset_version"].startswith("d10-")
print("dataset_version:", meta["dataset_version"])
for image in sample_visualizations(n=6):
    display(image)


## 4. E0 — zero-shot

Chạy cell, tải `backup-e0.zip`, rồi bấm **Save Version**.


In [ ]:
finish_kaggle.run_e0(force=False)
make_stage_backup("e0")


## 5. Kiểm tra mask loss và 220 visual token


In [ ]:
from PIL import Image
import gc
import torch

model_check, processor_check = lab_day10_solution.load_model_and_processor()
row = read_jsonl("data/splits/train.jsonl")[0]
with Image.open(image_path(row)) as source:
    image = source.convert("RGB")
answer = objects_to_answer(row["objects"])
encoded = encode_example(image, PROMPT_NORM1000, answer, processor_check)
assert vision_token_count(encoded) == 220
print("vision tokens:", vision_token_count(encoded))
target_ids = encoded["labels"][encoded["labels"] != -100]
print("Target decode:", processor_check.decode(target_ids, skip_special_tokens=False))
batch = collate_fn([encoded, encoded], processor_check.tokenizer.pad_token_id)
print({key: tuple(value.shape) for key, value in batch.items()})

del model_check, processor_check
gc.collect()
torch.cuda.empty_cache()


## 6. E1 — LoRA LLM

Sau khi inference và chấm điểm xong, tải `backup-e1.zip` và bấm **Save Version** trước khi chạy E2.


In [ ]:
finish_kaggle.run_sft_setup("E1", force=False)
make_stage_backup("e1")


## 7. E2 — LoRA LLM + projector

Runner giữ projector FP32 khi train, lưu `projector.pt`, sau đó nạp lại artifact cho inference. Tải `backup-e2.zip` và bấm **Save Version**.


In [ ]:
finish_kaggle.run_sft_setup("E2", force=False)
make_stage_backup("e2")


## 8. E3 — thêm LoRA ViT

Tải `backup-e3.zip` và bấm **Save Version** trước bước cuối.


In [ ]:
finish_kaggle.run_sft_setup("E3", force=False)
make_stage_backup("e3")


## 9. E4, reward test, báo cáo và ZIP cuối

Cell này tự bỏ qua E0–E3 đã hoàn tất, chạy E4 `data_size`, tạo diagnostic sai thang toạ độ, ảnh GT/prediction, `REPORT.md` và kiểm tra `0 FAIL`.


In [ ]:
print("Reward tests:", check_reward_cases())
final_archive = finish_kaggle.run_all(force=False)
display(FileLink(str(final_archive)))
print("Hãy tải ZIP và bấm Save Version.")


## 10. Khôi phục sau khi kernel restart

Nếu `/kaggle/working` còn repository, chỉ cần chạy lại cell 0 rồi cell của stage: runner sẽ skip phần hoàn tất hoặc tiếp tục inference JSONL. Nếu thư mục đã bị xoá, clone lại bằng cell 0, chuẩn bị dữ liệu, upload các `backup-e*.zip` đã tải và gọi `restore_stage_backup('/kaggle/working/backup-eN.zip')` cho từng file trước khi chạy tiếp.
